# Bài Kiểm Tra Trí Tuệ Nhân Tạo (Câu 2, 3, 4)

- **Họ và tên:** Nguyễn Trung Tín
- **MSSV:** 25110200091
- **Khoa:** CNTT-TTNT
- **Khóa:** 25

---
## Câu 2: Entropy, Information Gain và thuật toán ID3 (PlayTennis)

- **a)** Viết hàm `entropy(y)` và tính `Entropy(PlayTennis)`.
- **b)** Tính Information Gain (IG) của 4 thuộc tính (`Outlook`, `Temperature`, `Humidity`, `Wind`) và cho biết ID3 chọn thuộc tính nào làm nút gốc.
- **c)** Huấn luyện `DecisionTreeClassifier(criterion="entropy", random_state=42)` (mã hóa biến phân loại bằng `pd.get_dummies` / `OrdinalEncoder`), vẽ cây bằng `plot_tree`, và dự đoán mẫu `("Sunny", "Cool", "High", "Strong")`.

In [ ]:
import math
from collections import Counter
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeClassifier, plot_tree
# from sklearn.preprocessing import OrdinalEncoder  # Cách mã hóa thay thế

# Tập dữ liệu D
data = [
    ["Sunny", "Hot", "High", "Weak", "No"],
    ["Sunny", "Hot", "High", "Strong", "No"],
    ["Overcast", "Hot", "High", "Weak", "Yes"],
    ["Rain", "Mild", "High", "Weak", "Yes"],
    ["Rain", "Cool", "Normal", "Weak", "Yes"],
    ["Rain", "Cool", "Normal", "Strong", "No"],
    ["Overcast", "Cool", "Normal", "Strong", "Yes"],
    ["Sunny", "Mild", "High", "Weak", "No"],
    ["Sunny", "Cool", "Normal", "Weak", "Yes"],
    ["Rain", "Mild", "Normal", "Weak", "Yes"],
    ["Sunny", "Mild", "Normal", "Strong", "Yes"],
    ["Overcast", "Mild", "High", "Strong", "Yes"],
    ["Overcast", "Hot", "Normal", "Weak", "Yes"],
    ["Rain", "Mild", "High", "Strong", "No"],
]
columns = ["Outlook", "Temperature", "Humidity", "Wind", "PlayTennis"]
df = pd.DataFrame(data, columns=columns)

# a) Hàm tính Entropy và Entropy(PlayTennis)
def entropy(y):
    counts = Counter(y)
    n = len(y)
    return -sum((c / n) * math.log2(c / n) for c in counts.values() if c > 0)

h_play = entropy(df["PlayTennis"])
print(f"a) Entropy(PlayTennis) = {h_play:.6f}")

# b) Tính Information Gain cho cả 4 thuộc tính và chọn nút gốc
def information_gain(dataframe, attr, target="PlayTennis"):
    total_ent = entropy(dataframe[target])
    n = len(dataframe)
    weighted_ent = sum(
        (len(sub) / n) * entropy(sub[target])
        for _, sub in dataframe.groupby(attr)
    )
    return total_ent - weighted_ent

features = ["Outlook", "Temperature", "Humidity", "Wind"]
igs = {attr: information_gain(df, attr) for attr in features}
print("\nb) Information Gain của các thuộc tính:")
for attr, ig in igs.items():
    print(f"   IG({attr}) = {ig:.6f}")

root_attr = max(igs, key=igs.get)
print(f"-> Thuật toán ID3 chọn '{root_attr}' làm nút gốc (IG = {igs[root_attr]:.6f}).")

# c) Huấn luyện DecisionTreeClassifier với mã hóa biến phân loại, vẽ cây và dự đoán
# Ghi chú về cách mã hóa:
# - pd.get_dummies (One-Hot Encoding) tạo các cột nhị phân 0/1 cho từng giá trị,
#   tránh áp đặt thứ tự số giả tạo giữa các giá trị định danh (như Sunny, Overcast, Rain).
# - OrdinalEncoder ánh xạ từng giá trị thành số nguyên (0, 1, 2, ...), gọn nhẹ hơn
#   nhưng vô tình tạo ra thứ tự số giữa các nhãn không có tính thứ tự.
X = df[features]
y = df["PlayTennis"]

X_encoded = pd.get_dummies(X)

clf = DecisionTreeClassifier(criterion="entropy", random_state=42)
clf.fit(X_encoded, y)

plt.figure(figsize=(14, 8))
plot_tree(
    clf,
    feature_names=X_encoded.columns,
    class_names=clf.classes_,
    filled=True,
    rounded=True,
    fontsize=10,
)
plt.title("Cây quyết định (ID3 / Entropy) trên tập dữ liệu PlayTennis")
plt.show()

# Dự đoán mẫu ("Sunny", "Cool", "High", "Strong")
sample = pd.DataFrame(
    [["Sunny", "Cool", "High", "Strong"]],
    columns=features
)
sample_encoded = pd.get_dummies(sample).reindex(columns=X_encoded.columns, fill_value=0)
prediction = clf.predict(sample_encoded)[0]
print(f"\nc) Dự đoán cho mẫu ('Sunny', 'Cool', 'High', 'Strong'): {prediction}")

**Kết quả kiểm chứng Câu 2:**
- Tổng số mẫu: 14 (`Yes`: 9, `No`: 5)
- `Entropy(PlayTennis)` = **0.940286**
- Information Gain của 4 thuộc tính:
  - `IG(Outlook)` = **0.246750**
  - `IG(Humidity)` = **0.151836**
  - `IG(Wind)` = **0.048127**
  - `IG(Temperature)` = **0.029223**
- **Thuộc tính gốc (ID3):** `Outlook` (IG lớn nhất = 0.246750).
- **Dự đoán mẫu `("Sunny", "Cool", "High", "Strong")`:** `No`.

---
## Câu 3: Khảo sát hiện tượng Overfitting trên `load_breast_cancer`

- Chia dữ liệu train/test tỷ lệ 70/30 (`test_size=0.3, random_state=42`).
- **a)** Huấn luyện `DecisionTreeClassifier(random_state=42)` không giới hạn độ sâu; in độ chính xác trên tập huấn luyện (Train Accuracy) và tập kiểm thử (Test Accuracy).
- **b)** Chạy vòng lặp `max_depth` từ 1 đến 10 (`random_state=42`) và vẽ biểu đồ Train Accuracy vs Test Accuracy.
- **c)** Nhận xét về Overfitting và nêu 1 cách khắc phục.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

# Nạp dữ liệu và chia tập huấn luyện / kiểm thử theo tỷ lệ 70/30
X, y = load_breast_cancer(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

# a) Huấn luyện DecisionTreeClassifier không giới hạn độ sâu
clf_full = DecisionTreeClassifier(random_state=42)
clf_full.fit(X_train, y_train)

train_acc_full = accuracy_score(y_train, clf_full.predict(X_train))
test_acc_full = accuracy_score(y_test, clf_full.predict(X_test))

print("=== Câu 3a: Không giới hạn độ sâu (max_depth=None) ===")
print(f"Độ chính xác tập huấn luyện (Train Accuracy): {train_acc_full:.4f}")
print(f"Độ chính xác tập kiểm thử (Test Accuracy)   : {test_acc_full:.4f}")

# b) Chạy vòng lặp max_depth từ 1 đến 10
depths = list(range(1, 11))
train_accs = []
test_accs = []

print("\n=== Câu 3b: Độ chính xác theo max_depth (1..10) ===")
print(f"{'max_depth':>9} | {'Train Acc':>9} | {'Test Acc':>9}")
print("-" * 33)
for d in depths:
    clf = DecisionTreeClassifier(max_depth=d, random_state=42)
    clf.fit(X_train, y_train)
    tr_acc = accuracy_score(y_train, clf.predict(X_train))
    te_acc = accuracy_score(y_test, clf.predict(X_test))
    train_accs.append(tr_acc)
    test_accs.append(te_acc)
    print(f"{d:>9} | {tr_acc:>9.4f} | {te_acc:>9.4f}")

# Vẽ biểu đồ so sánh Train Accuracy và Test Accuracy
plt.figure(figsize=(8, 5))
plt.plot(depths, train_accs, marker="o", label="Độ chính xác tập huấn luyện (Train)")
plt.plot(depths, test_accs, marker="s", label="Độ chính xác tập kiểm thử (Test)")
plt.xticks(depths)
plt.xlabel("Độ sâu tối đa của cây (max_depth)")
plt.ylabel("Độ chính xác (Accuracy)")
plt.title("So sánh Train Accuracy và Test Accuracy theo max_depth (Breast Cancer)")
plt.legend()
plt.grid(True, linestyle="--", alpha=0.6)
plt.tight_layout()
plt.show()

### Câu 3c: Nhận xét
- **Overfitting là gì?** Overfitting (quá khớp) là hiện tượng mô hình học quá sát dữ liệu huấn luyện (bao gồm cả nhiễu và các đặc trưng ngẫu nhiên), dẫn đến độ chính xác trên tập train rất cao (gần hoặc bằng 100%) nhưng khả năng tổng quát hóa kém nên độ chính xác trên tập test thấp hơn rõ rệt.
- **Kết quả có cho thấy overfitting không?** **Có.** Ở câu 3a (không giới hạn độ sâu), cây đạt `Train Accuracy = 1.0000` nhưng `Test Accuracy` thấp hơn. Ở câu 3b, khi `max_depth` tăng cao, `Train Accuracy` tiếp tục tăng tiến tới `1.0000` trong khi `Test Accuracy` không tăng thêm (thậm chí giảm nhẹ sau điểm tối ưu), làm khoảng cách giữa hai đường ngày càng lớn.
- **1 cách khắc phục:** Giới hạn độ sâu tối đa của cây (`max_depth` — cắt tỉa trước / pre-pruning), chọn giá trị `max_depth` tại điểm cho `Test Accuracy` cao nhất mà chênh lệch với `Train Accuracy` còn nhỏ (ngoài ra có thể cắt tỉa bằng `min_samples_leaf`, `ccp_alpha`, hoặc dùng `RandomForestClassifier`).

---
### Câu 4: Phân tích và lựa chọn mô hình triển khai thực tế

**a) Nhận xét từ kết quả Câu 3b:**
- Dựa vào bảng kết quả và biểu đồ ở Câu 3b (sau khi chạy trên Colab), độ chính xác trên tập kiểm thử (**Test Accuracy**) đạt giá trị cao nhất tại **`max_depth = [điền giá trị, thường quanh 3–5]`**, với **`Test Accuracy = [điền số]`** (trong khi **`Train Accuracy = [điền số]`**).
- Hai đường **Train Accuracy** và **Test Accuracy** bắt đầu tách nhau ra rõ rệt từ khoảng **`max_depth = [điền giá trị]`** (lúc này `Train Accuracy = [điền số]` còn `Test Accuracy = [điền số]`).
- Từ ngưỡng này trở đi đến `max_depth = 10`, `Train Accuracy` tiếp tục tăng sát hoặc đạt `1.0000`, còn `Test Accuracy` chững lại hoặc giảm nhẹ xuống `[điền số]`, cho thấy cây bắt đầu học thuộc nhiễu (overfitting).

**b) Lựa chọn `max_depth` để triển khai tại phòng khám ở Vĩnh Long:**
- Em chọn **`max_depth = [điền giá trị tối ưu từ câu a]`** vì đây là điểm mô hình tổng quát hóa tốt nhất trên dữ liệu mới, tránh được tình trạng overfitting khi gặp bệnh nhân thực tế chưa từng thấy trong tập huấn luyện.
- Việc chọn cây có độ sâu vừa phải cũng giúp mô hình đơn giản, dễ giải thích cho bác sĩ tại phòng khám hơn so với một cây quá sâu và phức tạp.
- Trong bài toán sàng lọc ung thư vú, hậu quả của hai loại dự đoán sai rất khác nhau:
  - **False Negative (bỏ sót người bệnh – dự đoán lành tính nhưng thực tế có bệnh):** Hậu quả cực kỳ nghiêm trọng vì bệnh nhân mất cơ hội phát hiện sớm và điều trị kịp thời, làm bệnh tiến triển nặng hơn, thậm chí nguy hiểm đến tính mạng.
  - **False Positive (báo nhầm người khỏe – dự đoán có bệnh nhưng thực tế lành tính):** Gây tâm lý hoang mang cho bệnh nhân và tốn thêm chi phí, thời gian làm xét nghiệm chuyên sâu (sinh thiết, chụp chiếu lại), nhưng vẫn có thể kiểm tra lại để xác nhận an toàn.
- Vì vậy, khi triển khai hỗ trợ sàng lọc tại phòng khám ở Vĩnh Long, cần ưu tiên mô hình không bị overfitting và hạn chế tối đa **False Negative** (bỏ sót ca bệnh) hơn là chỉ nhìn vào độ chính xác tổng thể.